# Run the app on Kaggle (for the screenshots)

Settings: turn on **GPU T4** and **Internet**. Add your uploaded dataset (it holds `weights.zip` and `results.zip`) with **Add Input**.

Run the cells from top to bottom. The last cell prints a public link: open it in your browser and take the four screenshots.

## 1. Setup

In [ ]:
!pip install -q diffusers gradio rouge-score

In [ ]:
import os
for folder in ['starter', 'starter/data/uhat', 'part_a_mae', 'part_b_vae', 'part_c_diffusion', 'part_d_rag',
               'results', 'weights', 'cache', 'app']:
    os.makedirs(folder, exist_ok=True)

In [ ]:
import glob, os, shutil
for path in glob.glob('/kaggle/input/**/*.zip', recursive=True):
    os.system('unzip -q -o "' + path + '" -d .')
for folder in ['weights', 'results']:
    for found in glob.glob('/kaggle/input/**/' + folder, recursive=True):
        if os.path.isdir(found):
            shutil.copytree(found, folder, dirs_exist_ok=True)
print(sorted(os.listdir('weights')))
print(len(os.listdir('results')), 'files in results')

In [ ]:
!wget -q -O data.zip 'https://zenodo.org/records/3670611/files/data.zip?download=1'
!unzip -q -o data.zip -d starter/data/uhat

## 2. Files

In [ ]:
%%writefile starter/bioasq.py
"""Part D data: biomedical questions + PubMed abstracts (BioASQ, mini version)."""
from datasets import load_dataset

BIOASQ_REPO = "enelpol/rag-mini-bioasq"

def load_corpus():
    """Returns {passage_id (int): passage text (str)} for all 40k abstracts."""
    ds = load_dataset(BIOASQ_REPO, "text-corpus")
    split = list(ds.keys())[0]             # the corpus has a single split
    return {int(r["id"]): r["passage"] for r in ds[split] if r["passage"]}

def load_questions(split="test", corpus=None):
    """Returns a list of dicts: {qid, question, answer, relevant_ids}.
    relevant_ids = the ids of the abstracts that answer the question (the
    ground truth for measuring retrieval). If `corpus` is given, ids that are
    not in the corpus are dropped, and questions left with none are skipped."""
    ds = load_dataset(BIOASQ_REPO, "question-answer-passages", split=split)
    out = []
    for r in ds:
        rel = [int(i) for i in r["relevant_passage_ids"]]
        if corpus is not None:
            rel = [i for i in rel if i in corpus]
        if rel:
            out.append({"qid": int(r["id"]), "question": r["question"],
                        "answer": r["answer"], "relevant_ids": rel})
    return out


In [ ]:
%%writefile starter/eurosat.py
"""Part A data: EuroSAT RGB satellite tiles (64 x 64, 10 land-cover classes)."""
import numpy as np
import torch
from datasets import load_dataset

EUROSAT_REPO = "blanchon/EuroSAT_RGB"
CLASS_NAMES = ["AnnualCrop", "Forest", "HerbaceousVegetation", "Highway", "Industrial",
               "Pasture", "PermanentCrop", "Residential", "River", "SeaLake"]
PATCH = 8                      # 64 / 8 = 8 x 8 = 64 patches per image

def load_eurosat_split(split):
    """split in {'train', 'validation', 'test'}.
    Returns images (N, 3, 64, 64) float32 in [0, 1] and labels (N,) int64."""
    ds = load_dataset(EUROSAT_REPO, split=split)
    imgs = np.stack([np.asarray(im.convert("RGB"), dtype=np.uint8) for im in ds["image"]])
    images = torch.from_numpy(imgs).permute(0, 3, 1, 2).float() / 255.0
    labels = torch.tensor(ds["label"], dtype=torch.long)
    return images, labels

def few_label_indices(labels, per_class=16, seed=42):
    """Indices of a fixed labelled subset: `per_class` images of each class.
    Every group gets the same 160 images."""
    rng = np.random.default_rng(seed)
    idx = []
    for c in range(len(CLASS_NAMES)):
        members = np.flatnonzero(labels.numpy() == c)
        idx.extend(rng.choice(members, size=per_class, replace=False).tolist())
    return torch.tensor(sorted(idx))

def patchify(images, p=PATCH):
    """(N, C, H, W) -> (N, L, p*p*C) with L = (H/p)*(W/p), patches in row-major order."""
    N, C, H, W = images.shape
    x = images.reshape(N, C, H // p, p, W // p, p)
    x = x.permute(0, 2, 4, 3, 5, 1)                 # N, H/p, W/p, p, p, C
    return x.reshape(N, (H // p) * (W // p), p * p * C)

def unpatchify(patches, p=PATCH, C=3):
    """Inverse of patchify: (N, L, p*p*C) -> (N, C, H, W)."""
    N, L, _ = patches.shape
    h = w = int(L ** 0.5)
    x = patches.reshape(N, h, w, p, p, C).permute(0, 5, 1, 3, 2, 4)
    return x.reshape(N, C, h * p, w * p)


In [ ]:
%%writefile starter/uhat.py
"""Parts B and C data: UHaT handwritten Urdu characters (28 x 28, grayscale).
Unzip data.zip from Zenodo (or the Kaggle copy) into data/uhat/ first."""
from pathlib import Path
import numpy as np
import torch
from PIL import Image

IMG_EXT = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

def _find_split_dir(root, kind, split):
    """Find the folder whose name mentions both `kind` ('char'/'digit') and
    `split` ('train'/'test') and that contains one sub-folder per class."""
    for d in sorted(Path(root).rglob("*")):
        name = d.name.lower()
        if d.is_dir() and kind in name and split in name:
            if any(c.is_dir() for c in d.iterdir()):
                return d
    found = [str(d) for d in Path(root).rglob("*") if d.is_dir()][:20]
    raise FileNotFoundError(f"No '{kind}'/'{split}' folder under {root}. Folders seen: {found}")

def load_uhat_characters(root="data/uhat", split="train", class_names=None):
    """Returns images (N, 1, 28, 28) float32 in [0, 1] with WHITE strokes on a
    BLACK background, labels (N,) int64, and the list of class names.
    Pass the training class_names when loading 'test' so labels match."""
    split_dir = _find_split_dir(root, "char", split)
    if class_names is None:
        class_names = sorted(c.name.lower() for c in split_dir.iterdir() if c.is_dir())
    index = {name: i for i, name in enumerate(class_names)}
    images, labels = [], []
    for cdir in sorted(split_dir.iterdir()):
        if not cdir.is_dir() or cdir.name.lower() not in index:
            continue
        for f in sorted(cdir.iterdir()):
            if f.suffix.lower() in IMG_EXT:
                im = Image.open(f).convert("L").resize((28, 28))
                images.append(np.asarray(im, dtype=np.float32) / 255.0)
                labels.append(index[cdir.name.lower()])
    x = torch.from_numpy(np.stack(images)).unsqueeze(1)
    # Make polarity consistent: if most pixels are bright, the background is
    # white, so invert to get white strokes on black.
    flip = x.mean(dim=(1, 2, 3)) > 0.5
    x[flip] = 1.0 - x[flip]
    return x, torch.tensor(labels, dtype=torch.long), class_names


In [ ]:
%%writefile starter/viz.py
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch

def save_grid(images, path, nrow=8, titles=None):
    """images: (N, C, H, W) in [0, 1]. Saves an nrow-wide grid to `path`."""
    images = images.detach().cpu().clamp(0, 1)
    n = images.shape[0]
    rows = (n + nrow - 1) // nrow
    fig, axes = plt.subplots(rows, nrow, figsize=(1.4 * nrow, 1.4 * rows), squeeze=False)
    for i, ax in enumerate(axes.flat):
        ax.axis("off")
        if i < n:
            img = images[i].permute(1, 2, 0).numpy()
            ax.imshow(img.squeeze(-1) if img.shape[-1] == 1 else img, cmap="gray", vmin=0, vmax=1)
            if titles is not None:
                ax.set_title(titles[i], fontsize=7)
    fig.tight_layout()
    fig.savefig(path, dpi=120)
    plt.close(fig)


In [ ]:
%%writefile common.py
import os
import sys
import json
import time
import torch

sys.path.append("starter")

device = "cuda" if torch.cuda.is_available() else "cpu"
UHAT_ROOT = "starter/data/uhat"


def make_folders():
    os.makedirs("results", exist_ok=True)
    os.makedirs("weights", exist_ok=True)
    os.makedirs("cache", exist_ok=True)


def save_json(obj, path):
    file = open(path, "w")
    json.dump(obj, file, indent=1)
    file.close()


def load_json(path):
    file = open(path)
    obj = json.load(file)
    file.close()
    return obj


def mean(values):
    return sum(values) / len(values)


def count_params(model):
    total = 0
    for p in model.parameters():
        total = total + p.numel()
    return total


def gpu_name():
    if torch.cuda.is_available():
        return torch.cuda.get_device_name(0)
    return "cpu"


def batches(n, batch_size, shuffle=True):
    if shuffle:
        order = torch.randperm(n)
    else:
        order = torch.arange(n)
    for start in range(0, n, batch_size):
        yield order[start:start + batch_size]


def load_uhat():
    saved_file = "cache/uhat.pt"
    if os.path.exists(saved_file):
        return torch.load(saved_file)
    from uhat import load_uhat_characters
    x_train, y_train, names = load_uhat_characters(root=UHAT_ROOT, split="train")
    x_test, y_test, _ = load_uhat_characters(root=UHAT_ROOT, split="test", class_names=names)
    os.makedirs("cache", exist_ok=True)
    torch.save((x_train, y_train, x_test, y_test, names), saved_file)
    return x_train, y_train, x_test, y_test, names


In [ ]:
%%writefile part_a_mae/mae_model.py
import torch
import torch.nn as nn


class MAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.num_patches = 64
        self.num_keep = 16

        self.patch_embed = nn.Linear(192, 192)
        self.pos_embed = nn.Parameter(torch.randn(1, 64, 192) * 0.02)
        enc_layer = nn.TransformerEncoderLayer(d_model=192, nhead=3, dim_feedforward=768, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=6, enable_nested_tensor=False)
        self.enc_norm = nn.LayerNorm(192)

        self.dec_embed = nn.Linear(192, 128)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, 128))
        self.dec_pos_embed = nn.Parameter(torch.randn(1, 64, 128) * 0.02)
        dec_layer = nn.TransformerEncoderLayer(d_model=128, nhead=4, dim_feedforward=512, batch_first=True)
        self.decoder = nn.TransformerEncoder(dec_layer, num_layers=2, enable_nested_tensor=False)
        self.dec_out = nn.Linear(128, 192)

    def random_masking(self, batch_size, device):
        noise = torch.rand(batch_size, self.num_patches, device=device)
        order = noise.argsort(dim=1)
        keep_idx = order[:, :self.num_keep]
        mask = torch.ones(batch_size, self.num_patches, device=device)
        mask.scatter_(1, keep_idx, 0.0)
        return keep_idx, mask

    def forward(self, patches):
        batch_size = patches.shape[0]
        keep_idx, mask = self.random_masking(batch_size, patches.device)

        x = self.patch_embed(patches) + self.pos_embed
        index = keep_idx.unsqueeze(-1).expand(-1, -1, 192)
        visible = torch.gather(x, 1, index)
        encoded = self.enc_norm(self.encoder(visible))

        d = self.dec_embed(encoded)
        full = self.mask_token.expand(batch_size, self.num_patches, 128)
        index = keep_idx.unsqueeze(-1).expand(-1, -1, 128)
        full = full.scatter(1, index, d)
        full = full + self.dec_pos_embed
        full = self.decoder(full)
        pred = self.dec_out(full)

        loss = self.masked_loss(pred, patches, mask)
        return loss, pred, mask

    def masked_loss(self, pred, patches, mask):
        per_patch = ((pred - patches) ** 2).mean(dim=-1)
        return (per_patch * mask).sum() / mask.sum()

    def features(self, patches):
        x = self.patch_embed(patches) + self.pos_embed
        encoded = self.enc_norm(self.encoder(x))
        return encoded.mean(dim=1)


In [ ]:
%%writefile part_b_vae/vae_model.py
import torch
import torch.nn as nn
import torch.nn.functional as F


class VAE(nn.Module):
    def __init__(self, latent_dim=16):
        super().__init__()
        self.latent_dim = latent_dim

        self.enc_conv1 = nn.Conv2d(1, 32, 4, stride=2, padding=1)
        self.enc_conv2 = nn.Conv2d(32, 64, 4, stride=2, padding=1)
        self.fc_mu = nn.Linear(64 * 7 * 7, latent_dim)
        self.fc_logvar = nn.Linear(64 * 7 * 7, latent_dim)

        self.dec_fc = nn.Linear(latent_dim, 64 * 7 * 7)
        self.dec_conv1 = nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1)
        self.dec_conv2 = nn.ConvTranspose2d(32, 1, 4, stride=2, padding=1)

    def encode(self, x):
        h = F.relu(self.enc_conv1(x))
        h = F.relu(self.enc_conv2(h))
        h = h.flatten(start_dim=1)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + std * eps

    def decode(self, z):
        h = self.dec_fc(z)
        h = h.reshape(-1, 64, 7, 7)
        h = F.relu(self.dec_conv1(h))
        return self.dec_conv2(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        logits = self.decode(z)
        return logits, mu, logvar


def kl_term(mu, logvar):
    kl = -0.5 * (1 + logvar - mu ** 2 - logvar.exp()).sum(dim=1)
    return kl.mean()


def recon_term(logits, x):
    bce = F.binary_cross_entropy_with_logits(logits, x, reduction="none")
    return bce.sum(dim=(1, 2, 3)).mean()


def vae_loss(logits, x, mu, logvar):
    recon = recon_term(logits, x)
    kl = kl_term(mu, logvar)
    return recon + kl, recon, kl


In [ ]:
%%writefile part_c_diffusion/schedule.py
import torch

T = 1000

betas = torch.linspace(1e-4, 0.02, T)
alphas = 1.0 - betas
alpha_bars = torch.cumprod(alphas, dim=0)


def add_noise(x0, t, noise):
    abar = alpha_bars[t.cpu()].to(x0.device).reshape(-1, 1, 1, 1)
    return abar.sqrt() * x0 + (1 - abar).sqrt() * noise


In [ ]:
%%writefile part_c_diffusion/unet.py
from diffusers import UNet2DModel


def make_unet():
    unet = UNet2DModel(
        sample_size=28,
        in_channels=1,
        out_channels=1,
        layers_per_block=2,
        block_out_channels=(32, 64, 128),
        down_block_types=("DownBlock2D", "AttnDownBlock2D", "AttnDownBlock2D"),
        up_block_types=("AttnUpBlock2D", "AttnUpBlock2D", "UpBlock2D"),
        num_class_embeds=40,
    )
    return unet


In [ ]:
%%writefile part_c_diffusion/sample.py
import sys
sys.path.append(".")
sys.path.append("starter")
sys.path.append("part_c_diffusion")

import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from common import *
from viz import save_grid
from schedule import T, betas, alphas, alpha_bars, add_noise
from unet import make_unet


def sample_images(unet, labels, keep_steps=None):
    n = labels.shape[0]
    x = torch.randn(n, 1, 28, 28, device=device)
    saved = {}
    with torch.no_grad():
        for t in reversed(range(T)):
            t_batch = torch.full((n,), t, device=device, dtype=torch.long)
            predicted = unet(x, t_batch, class_labels=labels).sample
            alpha = alphas[t].item()
            abar = alpha_bars[t].item()
            beta = betas[t].item()
            x = (1 / alpha ** 0.5) * (x - (1 - alpha) / (1 - abar) ** 0.5 * predicted)
            if t > 0:
                x = x + (beta ** 0.5) * torch.randn_like(x)
            if keep_steps is not None and t in keep_steps:
                saved[t] = x.clamp(-1, 1).cpu()
    return x.clamp(-1, 1), saved


def to_01(x):
    return (x + 1) / 2


if __name__ == "__main__":
    torch.manual_seed(6)
    make_folders()

    x_train, y_train, x_test, y_test, names = load_uhat()

    unet = make_unet().to(device)
    unet.load_state_dict(torch.load("weights/unet.pt", map_location=device))
    unet.eval()

    image = x_train[0:1] * 2 - 1
    times = [0, 100, 250, 500, 750, 999]
    noised = []
    noise = torch.randn_like(image)
    for t in times:
        noised.append(add_noise(image, torch.tensor([t]), noise))
    noised = torch.cat(noised, dim=0).clamp(-1, 1)
    save_grid(to_01(noised), "results/fig_diffusion_forward.png", nrow=6,
              titles=["t=" + str(t) for t in times])
    print("saved results/fig_diffusion_forward.png")

    labels = torch.arange(40).repeat_interleave(8).to(device)
    all_samples = []
    for start in range(0, 320, 160):
        samples, saved = sample_images(unet, labels[start:start + 160])
        all_samples.append(samples.cpu())
        print("sampled", start + 160, "of 320")
    all_samples = to_01(torch.cat(all_samples, dim=0))

    fig, axes = plt.subplots(40, 8, figsize=(8, 40))
    for row in range(40):
        for col in range(8):
            ax = axes[row, col]
            ax.imshow(all_samples[row * 8 + col, 0], cmap="gray", vmin=0, vmax=1)
            ax.set_xticks([])
            ax.set_yticks([])
            if col == 0:
                ax.set_ylabel(names[row], rotation=0, ha="right", va="center", fontsize=8)
    plt.tight_layout()
    plt.savefig("results/fig_diffusion_samples.png", dpi=100)
    plt.close()
    print("saved results/fig_diffusion_samples.png")

    keep = [999, 750, 500, 250, 100, 0]
    one_label = torch.tensor([0], device=device)
    final, saved = sample_images(unet, one_label, keep_steps=keep)
    steps = torch.cat([saved[t] for t in keep], dim=0)
    save_grid(to_01(steps), "results/fig_diffusion_denoising.png", nrow=6,
              titles=["t=" + str(t) for t in keep])
    print("saved results/fig_diffusion_denoising.png (character: " + names[0] + ")")


In [ ]:
%%writefile part_d_rag/rag.py
import json
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"

RETRIEVER_NAME = "sentence-transformers/all-MiniLM-L6-v2"
GENERATOR_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
CHUNK_WORDS = 150
OVERLAP_WORDS = 30


def split_into_chunks(text, size=CHUNK_WORDS, overlap=OVERLAP_WORDS):
    words = text.split()
    step = size - overlap
    chunks = []
    start = 0
    while True:
        piece = words[start:start + size]
        chunks.append(" ".join(piece))
        if start + size >= len(words):
            break
        start = start + step
    return chunks


def load_retriever():
    tokenizer = AutoTokenizer.from_pretrained(RETRIEVER_NAME)
    model = AutoModel.from_pretrained(RETRIEVER_NAME)
    model = model.to(device)
    model.eval()
    return tokenizer, model


def embed_texts(texts, tokenizer, model, batch_size=128):
    all_vectors = []
    with torch.no_grad():
        for start in range(0, len(texts), batch_size):
            batch = texts[start:start + batch_size]
            encoded = tokenizer(batch, padding=True, truncation=True, max_length=256, return_tensors="pt")
            encoded = encoded.to(device)
            hidden = model(**encoded).last_hidden_state
            mask = encoded["attention_mask"].unsqueeze(-1).float()
            pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1)
            pooled = F.normalize(pooled, p=2, dim=1)
            all_vectors.append(pooled.cpu())
    return torch.cat(all_vectors, dim=0)


class Index:
    def __init__(self, folder="cache"):
        self.embeddings = torch.load(folder + "/embeddings.pt").to(device)
        file = open(folder + "/chunks.json", encoding="utf-8")
        data = json.load(file)
        file.close()
        self.chunk_abstract_ids = data["abstract_ids"]
        self.chunk_texts = data["texts"]

        self.abstract_ids = sorted(set(self.chunk_abstract_ids))
        position = {}
        for i, abstract_id in enumerate(self.abstract_ids):
            position[abstract_id] = i
        mapped = []
        for abstract_id in self.chunk_abstract_ids:
            mapped.append(position[abstract_id])
        self.chunk_to_abstract = torch.tensor(mapped, device=device)

    def top_abstracts(self, query_vector, k):
        scores = self.embeddings @ query_vector.to(device)
        best = torch.full((len(self.abstract_ids),), -10.0, device=device)
        best = best.scatter_reduce(0, self.chunk_to_abstract, scores, reduce="amax", include_self=True)
        top = best.topk(k).indices.tolist()
        return [self.abstract_ids[i] for i in top]

    def top_chunks(self, query_vector, k):
        scores = self.embeddings @ query_vector.to(device)
        top = scores.topk(k).indices.tolist()
        return top


In [ ]:
%%writefile part_d_rag/metrics.py
def hit_at_k(ranked, relevant, k):
    found = set(ranked[:k]) & set(relevant)
    if len(found) > 0:
        return 1.0
    return 0.0


def recall_at_k(ranked, relevant, k):
    relevant = set(relevant)
    found = set(ranked[:k]) & relevant
    return len(found) / min(k, len(relevant))


def mrr_at_10(ranked, relevant):
    relevant = set(relevant)
    for position, abstract_id in enumerate(ranked[:10]):
        if abstract_id in relevant:
            return 1.0 / (position + 1)
    return 0.0


In [ ]:
%%writefile part_d_rag/generate.py
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from rag import GENERATOR_NAME, device

NOTICE = "For educational use only. Not medical advice."


def load_generator():
    tokenizer = AutoTokenizer.from_pretrained(GENERATOR_NAME)
    model = AutoModelForCausalLM.from_pretrained(GENERATOR_NAME)
    if device == "cuda":
        model = model.half()
    model = model.to(device)
    model.eval()
    return tokenizer, model


def build_messages(question, passages):
    numbered = ""
    for i, passage in enumerate(passages):
        numbered = numbered + "[" + str(i + 1) + "] " + passage + "\n\n"

    system = ("You answer biomedical research questions. Use only the numbered passages given by the user. "
              "Cite the passage numbers in square brackets, for example [1]. "
              "If the passages do not contain the answer, say that you do not know.")
    user = "Passages:\n\n" + numbered + "Question: " + question
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]


def generate_answer(question, passages, tokenizer, model):
    messages = build_messages(question, passages)
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=256, do_sample=False)
    new_tokens = output[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


In [ ]:
%%writefile part_d_rag/build_index.py
import sys
sys.path.append(".")
sys.path.append("starter")
sys.path.append("part_d_rag")

import json
import time
import torch
from common import *
from bioasq import load_corpus, load_questions
from rag import split_into_chunks, load_retriever, embed_texts

make_folders()

corpus = load_corpus()
questions = load_questions("test", corpus)
print("abstracts:", len(corpus), "| test questions:", len(questions))

chunk_abstract_ids = []
chunk_texts = []
for abstract_id in sorted(corpus.keys()):
    for piece in split_into_chunks(corpus[abstract_id]):
        chunk_abstract_ids.append(abstract_id)
        chunk_texts.append(piece)
print("chunks:", len(chunk_texts))

longest = 0
for piece in chunk_texts:
    longest = max(longest, len(piece.split()))
print("longest chunk in words:", longest)

tokenizer, model = load_retriever()
start_time = time.time()
embeddings = embed_texts(chunk_texts, tokenizer, model)
seconds = time.time() - start_time
print("embedding matrix:", tuple(embeddings.shape), "| time in minutes:", round(seconds / 60, 1))

torch.save(embeddings, "cache/embeddings.pt")
file = open("cache/chunks.json", "w", encoding="utf-8")
json.dump({"abstract_ids": chunk_abstract_ids, "texts": chunk_texts}, file)
file.close()

total_relevant = 0
for q in questions:
    total_relevant = total_relevant + len(set(q["relevant_ids"]))

save_json({"abstracts": len(corpus), "chunks": len(chunk_texts), "test_questions": len(questions),
           "mean_relevant": total_relevant / len(questions), "embedding_seconds": seconds,
           "gpu": gpu_name()}, "results/rag_index_info.json")
print("saved cache/embeddings.pt, cache/chunks.json and results/rag_index_info.json")


In [ ]:
%%writefile app/app.py
import sys
sys.path.append(".")
sys.path.append("starter")
sys.path.append("part_a_mae")
sys.path.append("part_b_vae")
sys.path.append("part_c_diffusion")
sys.path.append("part_d_rag")

import numpy as np
import torch
import torch.nn as nn
import gradio as gr
from common import *
from eurosat import load_eurosat_split, patchify, unpatchify, CLASS_NAMES
from mae_model import MAE
from vae_model import VAE
from unet import make_unet
from sample import sample_images, to_01
from rag import load_retriever, embed_texts, Index
from generate import load_generator, generate_answer, NOTICE

store = {}


def get(name, loader):
    if name not in store:
        store[name] = loader()
    return store[name]


def to_picture(images, ncols, scale):
    images = images.detach().cpu().clamp(0, 1)
    n, c, h, w = images.shape
    rows = (n + ncols - 1) // ncols
    canvas = np.zeros((rows * h, ncols * w, 3), dtype=np.float32)
    for i in range(n):
        r = i // ncols
        col = i % ncols
        tile = images[i].permute(1, 2, 0).numpy()
        if c == 1:
            tile = np.repeat(tile, 3, axis=2)
        canvas[r * h:(r + 1) * h, col * w:(col + 1) * w] = tile
    canvas = np.repeat(np.repeat(canvas, scale, axis=0), scale, axis=1)
    return (canvas * 255).astype(np.uint8)


def load_mae_parts():
    model = MAE().to(device)
    model.load_state_dict(torch.load("weights/mae.pt", map_location=device))
    model.eval()
    classifier = nn.Linear(192, 10).to(device)
    classifier.load_state_dict(torch.load("weights/mae_classifier.pt", map_location=device))
    images, labels = load_eurosat_split("test")
    return model, classifier, images, labels


def run_mae(tile_number):
    model, classifier, images, labels = get("mae", load_mae_parts)
    tile_number = int(tile_number)
    image = images[tile_number:tile_number + 1]
    patches = patchify(image)
    with torch.no_grad():
        loss, pred, mask = model(patches.to(device))
        features = model.features(patches.to(device))
        predicted_class = classifier(features).argmax(dim=1).item()
    pred = pred.cpu()
    mask3 = mask.cpu().unsqueeze(-1)
    masked_input = unpatchify(patches * (1 - mask3) + 0.5 * mask3)
    reconstruction = unpatchify(patches * (1 - mask3) + pred * mask3)
    text = "Predicted class: " + CLASS_NAMES[predicted_class] + "  |  True class: " + CLASS_NAMES[labels[tile_number].item()]
    return to_picture(masked_input, 1, 4), to_picture(reconstruction, 1, 4), to_picture(image, 1, 4), text


def load_vae_parts():
    model = VAE().to(device)
    model.load_state_dict(torch.load("weights/vae.pt", map_location=device))
    model.eval()
    x_train, y_train, x_test, y_test, names = load_uhat()
    return model, x_test, y_test, names


def new_characters():
    model, x_test, y_test, names = get("vae", load_vae_parts)
    z = torch.randn(64, 16).to(device)
    with torch.no_grad():
        samples = torch.sigmoid(model.decode(z))
    return to_picture(samples, 8, 3)


def interpolate(first_name, second_name, amount):
    model, x_test, y_test, names = get("vae", load_vae_parts)
    first = names.index(first_name)
    second = names.index(second_name)
    a = x_test[(y_test == first).nonzero()[0].item()].unsqueeze(0).to(device)
    b = x_test[(y_test == second).nonzero()[0].item()].unsqueeze(0).to(device)
    with torch.no_grad():
        mu_a, _ = model.encode(a)
        mu_b, _ = model.encode(b)
        z = (1 - amount) * mu_a + amount * mu_b
        result = torch.sigmoid(model.decode(z))
    return to_picture(torch.cat([a.cpu(), result.cpu(), b.cpu()], dim=0), 3, 8)


def load_unet():
    unet = make_unet().to(device)
    unet.load_state_dict(torch.load("weights/unet.pt", map_location=device))
    unet.eval()
    return unet


def write_character(name):
    unet = get("unet", load_unet)
    names = load_json("results/diffusion_train.json")["class_names"]
    label = torch.full((8,), names.index(name), device=device, dtype=torch.long)
    samples, saved = sample_images(unet, label)
    return to_picture(to_01(samples), 8, 4)


def load_rag_parts():
    retriever_tokenizer, retriever_model = load_retriever()
    index = Index()
    generator_tokenizer, generator_model = load_generator()
    return retriever_tokenizer, retriever_model, index, generator_tokenizer, generator_model


def answer_question(question):
    retriever_tokenizer, retriever_model, index, generator_tokenizer, generator_model = get("rag", load_rag_parts)
    vector = embed_texts([question], retriever_tokenizer, retriever_model)[0]
    chunk_ids = index.top_chunks(vector, 5)
    passages = []
    for chunk_id in chunk_ids:
        passages.append(index.chunk_texts[chunk_id])
    answer = generate_answer(question, passages, generator_tokenizer, generator_model)

    shown = ""
    for number, chunk_id in enumerate(chunk_ids):
        shown = shown + "**[" + str(number + 1) + "]** abstract id " + str(index.chunk_abstract_ids[chunk_id]) + "\n\n"
        shown = shown + passages[number] + "\n\n"
    return answer + "\n\n*" + NOTICE + "*", shown


vae_names = load_json("results/vae_train.json")["class_names"] if os.path.exists("results/vae_train.json") else ["alif"]
diffusion_names = vae_names

with gr.Blocks(title="Four generative models") as demo:
    gr.Markdown("# Four generative models in one app")

    with gr.Tab("MAE (satellite images)"):
        gr.Markdown("Pick a test tile. The model sees only 25% of the patches and rebuilds the rest.")
        tile = gr.Slider(0, 5399, value=0, step=1, label="Test tile number")
        mae_button = gr.Button("Run")
        with gr.Row():
            masked_out = gr.Image(label="Masked input")
            recon_out = gr.Image(label="Reconstruction")
            original_out = gr.Image(label="Original")
        mae_text = gr.Textbox(label="Land cover class")
        mae_button.click(run_mae, inputs=tile, outputs=[masked_out, recon_out, original_out, mae_text])

    with gr.Tab("VAE (Urdu characters)"):
        vae_button = gr.Button("New random characters")
        vae_out = gr.Image(label="64 new characters")
        vae_button.click(new_characters, outputs=vae_out)
        with gr.Row():
            first_choice = gr.Dropdown(vae_names, value=vae_names[0], label="First character")
            second_choice = gr.Dropdown(vae_names, value=vae_names[1], label="Second character")
        amount = gr.Slider(0, 1, value=0.5, step=0.1, label="Morph amount")
        morph_out = gr.Image(label="First, morph, second")
        amount.change(interpolate, inputs=[first_choice, second_choice, amount], outputs=morph_out)

    with gr.Tab("Diffusion (write a character)"):
        character = gr.Dropdown(diffusion_names, value=diffusion_names[0], label="Character to write")
        diffusion_button = gr.Button("Generate 8 samples")
        diffusion_out = gr.Image(label="8 new samples")
        diffusion_button.click(write_character, inputs=character, outputs=diffusion_out)

    with gr.Tab("RAG (biomedical questions)"):
        gr.Markdown("**" + NOTICE + "**")
        question_box = gr.Textbox(label="Your question")
        rag_button = gr.Button("Answer")
        answer_out = gr.Markdown()
        passages_out = gr.Markdown()
        rag_button.click(answer_question, inputs=question_box, outputs=[answer_out, passages_out])

if __name__ == "__main__":
    demo.launch()


## 3. Build the retrieval index (about 4 minutes)

In [ ]:
!python part_d_rag/build_index.py

## 4. Start the app

When the link `https://....gradio.live` appears, open it in your browser. Keep this cell running while you take the screenshots.

In [ ]:
!sed -i 's/demo.launch()/demo.launch(share=True)/' app/app.py
!python app/app.py